In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
pio.templates.default = "plotly_white"
import xarray as xr
from whakaaribn.visualize import trellis_plot
from whakaaribn import get_color

In [ ]:
try:
    data_file = snakemake.input.data
    forecast_all_data = snakemake.input.forecast_all_data
    forecast_seismic = snakemake.input.forecast_seismic
    forecast_gas = snakemake.input.forecast_gas
    forecast_all_data_hindcast = snakemake.input.forecast_all_data_hindcast
    forecast_seismic_hindcast = snakemake.input.forecast_seismic_hindcast
    forecast_gas_hindcast = snakemake.input.forecast_gas_hindcast
    grid_search_results = snakemake.input.grid_search_results
    forecast_sensitivity = snakemake.input.forecast_sensitivity
    forecast_uncertainty = snakemake.input.forecast_uncertainty
    pio.get_chrome()
except NameError:
    data_file = '../data/whakaari_data_with_groups.csv'
    forecast_all_data = '../forecasts/whakaari_forecasts.nc'
    forecast_seismic = '../forecasts/whakaari_forecasts_seismic.nc'
    forecast_gas = '../forecasts/whakaari_forecasts_gas.nc'
    forecast_all_data_hindcast = '../forecasts/whakaari_forecasts_all_data_hindcast.nc'
    forecast_seismic_hindcast = '../forecasts/whakaari_forecasts_seismic_hindcast.nc'
    forecast_gas_hindcast = '../forecasts/whakaari_forecasts_gas_hindcast.nc'
    grid_search_results = '../results/grid_search_results.csv'
    forecast_sensitivity = '../forecasts/whakaari_sensitivity.nc'
    forecast_uncertainty = '../forecasts/whakaari_uncertainty.nc'

In [ ]:
xds_best = xr.open_dataset(forecast_all_data)
data = pd.read_csv(data_file, parse_dates=True, index_col=0)

In [ ]:
xds_best = xr.open_dataset(forecast_all_data)
xds_best_seismic = xr.open_dataset(forecast_seismic)
xds_best_gas = xr.open_dataset(forecast_gas)

models = {'Eruption Probability (best model)': {'model': xds_best['probs'], 'color': get_color(0)},
          'Gas Eruption Probability': {'model': xds_best_gas['probs'], 'color': get_color(3, alpha=0.5)},
          'Seismic Eruption Probability': {'model': xds_best_seismic['probs'], 'color': get_color(4, alpha=0.5)}
          }
fig = trellis_plot(models, data)
try:
    fig.write_image(snakemake.output.eruption_forecasts, width=1200, height=1000, scale=5)
except NameError:
    pass
fig


In [ ]:
search_results = grid_search_results = pd.read_csv(grid_search_results,
                                                   index_col=(0, 1, 2),
                                                   converters={"params": eval})
fig = go.Figure()
metric = 'mod_roc_auc_no_pew'
nstates = search_results.index.get_level_values('nstates').unique()
pews = search_results.index.get_level_values('pew').unique()
for _nstates in nstates: 
    vals = []
    error = []
    showlegend=True
    for pew in pews:
        sdf_tmp = search_results.loc[(pew, _nstates)] 
        sdf_tmp = sdf_tmp.sort_values(by=[f'rank_test_{metric}'])
        vals.append(sdf_tmp.iloc[0][f'mean_test_{metric}'])
        error.append(sdf_tmp.iloc[0][f'std_test_{metric}'])
    vals = np.array(vals)
    error = np.array(error)
    fig.add_trace(go.Scatter(x=list(pews), y=vals, mode='lines',
                             name=f'{_nstates} states', line_color=get_color(_nstates-3)))
    fig.add_trace(go.Scatter(x=list(pews), y=vals+error, mode='lines', marker=dict(color="#444"),
                                 line=dict(width=0), showlegend=False))
    fig.add_trace(go.Scatter(x=list(pews), y=vals-error, mode='lines', marker=dict(color="#444"),
                                 line=dict(width=0), showlegend=False, fillcolor=get_color(_nstates-3, alpha=0.1),
                                 fill='tonexty'))
 
fig.update_layout(xaxis_title="Pre-eruption window", yaxis_title='AUC-ROC')
try:
    fig.write_image(snakemake.output.model_objective_function, width=900, height=400, scale=5)
except NameError:
    pass
fig

In [ ]:
xds_all = xr.open_dataset(forecast_sensitivity)

In [ ]:
median_model = xds_all.median('model').to_array()
min_model = xds_all.chunk(dict(model=-1)).quantile(0.15, 'model').to_array()
max_model = xds_all.chunk(dict(model=-1)).quantile(0.85, 'model').to_array()
models = {"Eruption Probability (median model)": {'model': median_model.squeeze('variable'), 'color': get_color(0)},
          "Eruption Probability (best model)": {'model': xds_best['probs'], 'color': get_color(1)},
          "min": {'model': min_model.squeeze('variable'), 'color': get_color(0, alpha=0.3)},
          "max": {'model': max_model.squeeze('variable'), 'color': get_color(0, alpha=0.3)}
}

fig = trellis_plot(models, data, plot_uncertainty='quantile')
try:
    fig.write_image(snakemake.output.forecast_sensitivity_plot, width=1200, height=1000, scale=5)
except NameError:
    pass
fig

In [ ]:
xds_all_1 = xr.open_dataset(forecast_uncertainty)

In [ ]:
median_model = xds_all_1.median('model_score').to_array()
# min_model = xds_all_1.chunk(dict(model_score=-1)).quantile(0.15, 'model_score')
# max_model = xds_all_1.chunk(dict(model_score=-1)).quantile(0.85, 'model_score')
min_model = xds_all_1.chunk(dict(model_score=-1)).min('model_score').to_array()
max_model = xds_all_1.chunk(dict(model_score=-1)).max('model_score').to_array()
models = {"Eruption Probability (median model)": {'model': median_model.squeeze('variable'), 'color': get_color(0)},
          "Eruption Probability (best model)": {'model': xds_best['probs'], 'color': get_color(1)},
          "min": {'model': min_model.squeeze('variable'), 'color': get_color(0, alpha=0.3)},
          "max": {'model': max_model.squeeze('variable'), 'color': get_color(0, alpha=0.3)}
}
fig = trellis_plot(models, data, plot_uncertainty='quantile')
try:
    fig.write_image(snakemake.output.forecast_uncertainty_plot, width=1200, height=1000, scale=5)
except NameError:
    pass
fig

In [ ]:
median_model = xds_all_1.median('model_score').to_array()
models = {"Eruption Probability (median model)": {'model': median_model.squeeze('variable'), 'color': get_color(0)},
          "Eruption Probability (best model)": {'model': xds_best['probs'], 'color': get_color(1)},
          "ensemble": {'model': xds_all_1.to_array().squeeze('variable'), 'colorscale': "algae"},
}
fig = trellis_plot(models, data, plot_uncertainty='ensemble')
try:
    fig.write_image(snakemake.output.forecast_ensemble_plot, width=1200, height=1000, scale=5)
except NameError:
    pass
fig

In [ ]:
xds_best = xr.open_dataset(forecast_all_data_hindcast)
xds_best_seismic = xr.open_dataset(forecast_seismic_hindcast)
xds_best_gas = xr.open_dataset(forecast_gas_hindcast)


models = {'Eruption Probability (best model)': {'model': xds_best['probs'], 'color': get_color(0)},
          'Gas Eruption Probability': {'model': xds_best_gas['probs'], 'color': get_color(3, alpha=0.5)},
          'Seismic Eruption Probability': {'model': xds_best_seismic['probs'], 'color': get_color(4, alpha=0.5)}
          }
fig = trellis_plot(models, data)
try:
    fig.write_image(snakemake.output.eruption_forecasts_hindcast, width=1200, height=1000, scale=5)
except NameError:
    pass
fig


In [ ]:
xds_best_gas.probs

In [ ]:
xds_best.probs